In [1]:
import sys
sys.path.append('../')
import time

import numpy as np
import matplotlib.pyplot as plt

import epics
from siriuspy.devices import CAXCtrl
from caxscripts.h5file import HDF5File

In [ ]:
cax = CAXCtrl()

# Functions

Useful functions for the scan

In [53]:

#todo: move to initial position

def current_position():
    return [cax.slit_A1.top_pos,
            cax.slit_A1.bottom_pos,
            cax.slit_A1.left_pos,
            cax.slit_A1.right_pos]

def open_all_slits():
    cax.slit_A1.move_robust_top(value=19.7-0.04)
    cax.slit_A1.move_robust_bottom(value=35.8)
    cax.slit_A1.move_robust_left(value=43.6-0.04)
    cax.slit_A1.move_robust_right(value=47.2)

def move_all(top_pos,bottom_pos,left_pos,right_pos):
    cax.slit_A1.move_top(value=top_pos)
    cax.slit_A1.move_bottom(value=bottom_pos)
    cax.slit_A1.move_left(value=left_pos)
    cax.slit_A1.move_right(value=right_pos)

def move_robust_all(top_pos,bottom_pos,left_pos,right_pos):
    cax.slit_A1.move_robust_top(value=top_pos)
    cax.slit_A1.move_robust_bottom(value=bottom_pos)
    cax.slit_A1.move_robust_left(value=left_pos)
    cax.slit_A1.move_robust_right(value=right_pos)

# Scan

## Parameters

In [44]:
MAXERRORCOUNT = 5

In [45]:
top_pos_open = 19.7 - 0.04
bottom_pos_open = 35.8
left_pos_open = 43.6 - 0.04
right_pos_open = 47.2

rangex = 1.3 + 0.1
rangey = 4.8

proport = rangey/rangex

In [5]:
L = 0.4

Ndxs = 7
Ndys = int(proport*Ndxs)
dxs = np.linspace(0,rangex-L,Ndxs)
dys = np.linspace(0,rangey-L,Ndys)

In [6]:
print('N of points:',Ndys*Ndxs)
# print(f'scan time estimative: {Ndys*Ndxs*4/60:.3f} min')

N of points: 161


# Execution

Initial state

In [27]:
local_time = time.localtime()
formatted_time = time.strftime("%Y%m%d-%H%M%S", local_time)
formatted_date = time.strftime("%Y%m%d", local_time)

formatted_time, formatted_date

('20250829-094858', '20250829')

In [26]:
top0, bottom0, left0, right0 = current_position()

with open(f'initial_position_slit_{formatted_time}.txt','w') as f:
    f.write(f'top: {top0}\n')
    f.write(f'bottom: {bottom0}\n')
    f.write(f'left: {left0}\n')
    f.write(f'right: {right0}')

print(top0, bottom0, left0, right0)

16.582421875 35.365625 42.9 47.07


Initializate file

In [49]:
filename = f'square_scan_{formatted_date}_01.h5'
filedir = f"/home/ids/data/{formatted_date}-Slit-Mirror"
file = HDF5File(filename=filename,filedir=filedir)

file.save_metadata(metadata_dict={
    'exposure_time': cax.dvf_B1.exposure_time,
    'slit_top': cax.slit_A1.top_pos,
    'slit_bottom': cax.slit_A1.bottom_pos,
    'slit_left': cax.slit_A1.left_pos,
    'slit_right': cax.slit_A1.right_pos
})

Loop

In [ ]:
t0 = time.time()

for i, dy in enumerate(dys):

    move_robust_all(top_pos    = top_pos_open-rangey+L+dy,
                    bottom_pos = bottom_pos_open-dy,
                    left_pos   = left_pos_open,
                    right_pos  = right_pos_open-rangex+L)

    for j, dx in enumerate(dxs):

        print(f'scan step x {j+1}/{len(dxs)} and step y {i+1}/{len(dys)}')

        cax.slit_A1.move_robust_left(value=left_pos_open-dx)
        cax.slit_A1.move_robust_right(value=right_pos_open-rangex+L+dx)

        count = 0
        while count < MAXERRORCOUNT:
            try:
                img1 = cax.dvf_A1.image
                count = MAXERRORCOUNT
            except Exception as err:
                print(f" WARNING. When trying to fetch image from DVF1: {err} ")
                time.sleep(2)
                count += 1
                if count < MAXERRORCOUNT:
                    print("\n Repeating the procedure...\n")
                else:
                    raise Exception("Client exception")

        count = 0
        while count < MAXERRORCOUNT:
            try:
                img2 = cax.dvf_B1.image
                count = MAXERRORCOUNT
            except Exception as err:
                print(f" WARNING. When trying to fetch image from DVF2: {err} ")
                time.sleep(2)
                count += 1
                if count < MAXERRORCOUNT:
                    print("\n Repeating the procedure...\n")
                else:
                    raise Exception("Client exception")

        pos_dict = {
            'slit_top'    : cax.slit_A1.top_pos,
            'slit_bottom' : cax.slit_A1.bottom_pos,
            'slit_left'   : cax.slit_A1.left_pos,
            'slit_right'  : cax.slit_A1.right_pos
        }

        grpname = f'scan-{i:02d}-{j:02d}'
        file.save_group(grpname=grpname, grpmetadata=pos_dict)
        file.save_dataset(dsetname=f'dvf1-img', grpname=grpname, dsetdata=img1)
        file.save_dataset(dsetname=f'dvf2-img', grpname=grpname, dsetdata=img2)

move_robust_all(top_pos    = top0,
                bottom_pos = bottom0,
                left_pos   = left0,
                right_pos  = right0)

t1 = time.time()
print(f'ellapsed time [s]: {t1-t0}')

Done!
Done!
Done!is Moving... | New pos: 43.56 | Curr: 43.56 | Dif: 0.0
Done!is Moving... | New pos: 46.2 | Curr: 46.20 | Dif: 0.0
scan step x 1/7 and step y 1/23
Done!
Done!
scan step x 2/7 and step y 1/23
Done!is Moving... | New pos: 43.39333333333334 | Curr: 43.39 | Dif: 2.6041666664866625e-05
Done!is Moving... | New pos: 46.36666666666667 | Curr: 46.37 | Dif: 2.6041666664866625e-05
scan step x 3/7 and step y 1/23
Done!is Moving... | New pos: 43.22666666666667 | Curr: 43.23 | Dif: 2.6041666664866625e-05
Done!is Moving... | New pos: 46.53333333333334 | Curr: 46.53 | Dif: 2.6041666664866625e-05
scan step x 4/7 and step y 1/23
Done!is Moving... | New pos: 43.06 | Curr: 43.06 | Dif: 0.0
Done!is Moving... | New pos: 46.7 | Curr: 46.70 | Dif: 0.0
scan step x 5/7 and step y 1/23
Done!is Moving... | New pos: 42.89333333333334 | Curr: 42.89 | Dif: 2.6041666664866625e-05
Done!is Moving... | New pos: 46.86666666666667 | Curr: 46.87 | Dif: 2.6041666664866625e-05
scan step x 6/7 and step y 1/23


CA.Client.Exception...............................................
    Context: "10.30.14.19:33461"
    Source File: ../cac.cpp line 1237
    Current Time: Fri Aug 29 2025 10:28:14.814806222
..................................................................


Done!is Moving... | New pos: 46.53333333333334 | Curr: 46.53 | Dif: 2.6041666664866625e-05
 WARNING. When trying to fetch image from DVF1: 'NoneType' object cannot be interpreted as an integer 

 Repeating the procedure...

scan step x 4/7 and step y 3/23
Done!is Moving... | New pos: 43.06 | Curr: 43.06 | Dif: 0.0
Done!is Moving... | New pos: 46.7 | Curr: 46.70 | Dif: 0.0
scan step x 5/7 and step y 3/23
Done!is Moving... | New pos: 42.89333333333334 | Curr: 42.89 | Dif: 2.6041666664866625e-05
Done!is Moving... | New pos: 46.86666666666667 | Curr: 46.87 | Dif: 2.6041666664866625e-05
scan step x 6/7 and step y 3/23
Done!is Moving... | New pos: 42.72666666666667 | Curr: 42.73 | Dif: 2.6041666664866625e-05
Done!is Moving... | New pos: 47.03333333333334 | Curr: 47.03 | Dif: 2.6041666664866625e-05
scan step x 7/7 and step y 3/23
Done!is Moving... | New pos: 42.56 | Curr: 42.56 | Dif: 0.0
Done!is Moving... | New pos: 47.2 | Curr: 47.20 | Dif: 0.0
Done!is Moving... | New pos: 15.86 | Curr: 15.

In [51]:
move_robust_all(top_pos=16.16,
                bottom_pos=34.9,
                left_pos=43.06,
                right_pos=46.7)

Done!is Moving... | New pos: 16.16 | Curr: 16.16 | Dif: 0.0
Done!is Moving... | New pos: 34.9 | Curr: 34.90 | Dif: 7.105427357601002e-15
Done!is Moving... | New pos: 43.06 | Curr: 43.06 | Dif: 0.0
Done!is Moving... | New pos: 46.7 | Curr: 46.70 | Dif: 0.0


In [54]:
move_robust_all(top_pos=16.86,
                bottom_pos=34.2,
                left_pos=43.39,
                right_pos=46.37)

Done!
Done!
Done!
Done!


In [55]:
move_all(top_pos=16.86,
         bottom_pos=34.2,
         left_pos=43.39+0.1,
         right_pos=46.37-0.1)

In [59]:
# LAST VALUES #

move_all(
    bottom_pos = 34.6,
    left_pos = 42.893359375,
    right_pos = 46.866640625,
    top_pos = 16.46
)

In [58]:
open_all_slits()

Done!is Moving... | New pos: 19.66 | Curr: 19.66 | Dif: 0.09367187499999972
Done!is Moving... | New pos: 35.8 | Curr: 35.80 | Dif: 0.0
Done!is Moving... | New pos: 43.56 | Curr: 43.56 | Dif: 0.0
Done!is Moving... | New pos: 47.2 | Curr: 47.20 | Dif: 0.0
